In [1]:
import pandas as pd
import glob
import os

path_to_csvs = "data/raw/*.csv" 
csv_files = glob.glob(path_to_csvs)

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    print("=" * 50)
    print(f"INSIGHTS FOR FILE: {file_name}")
    print("=" * 50)

    df = pd.read_csv(file_path)

    print(f"1. Dataset Shape (Rows, Columns): {df.shape}")
    print("-" * 30)
    print("2. Column Data Types:")
    print(df.dtypes)
    print("-" * 30)

    print("3. First 5 Rows of Data:")
    print(df.head())
    print("\n" + "#" * 50 + "\n")

In [2]:
import pandas as pd
import glob

csv_files = glob.glob("../data/raw/*.csv")
for file in csv_files:
    df = pd.read_csv(file)
    print(f"--- File: {file} ---")
    print("Shape:", df.shape)
    print("Data Types:\n", df.dtypes)
    print("Head:\n", df.head(), "\n")
    

--- File: ../data/raw\01_fund_master.csv ---
Shape: (40, 15)
Data Types:
 amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object
Head:
    amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regula

In [1]:
import requests
import pandas as pd
import json
import os

# Create raw directory if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

def fetch_and_save_nav(scheme_code, scheme_name=None):
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    print(f"Fetching data for: {scheme_name or scheme_code} (Code: {scheme_code})...")
    
    try:
        # 1. Send the network request
        response = requests.get(url, timeout=10)
        
        # Check if the server returned an error page (like a 404 or 500)
        if response.status_code != 200:
            print(f"❌ Server Error: Received status code {response.status_code}")
            return False
            
        # 2. Safely parse JSON structure
        try:
            payload = response.json()
        except json.JSONDecodeError:
            print("❌ Parsing Error: The API did not return valid JSON data. It might be down or rate-limiting.")
            return False
            
        # 3. Check if 'data' array exists in response
        if 'data' not in payload or not payload['data']:
            print(f"⚠️ Warning: No 'data' key found for code {scheme_code}. Response message: {payload.get('status', 'Unknown structure')}")
            return False
            
        # 4. Extract arrays and transform to Pandas DataFrame
        nav_list = payload['data']
        df = pd.DataFrame(nav_list)
        
        # Ensure data types are uniform before saving
        df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
        
        # Add tracking data columns
        df['amfi_code'] = scheme_code
        if scheme_name:
            df['scheme_name'] = scheme_name
            filename = f"data/raw/nav_{scheme_name}_{scheme_code}.csv"
        else:
            filename = f"data/raw/hdfc_{scheme_code}_raw.csv"
            
        # Save DataFrame to storage file
        df.to_csv(filename, index=False)
        print(f"✅ Success! Saved {len(df)} historical rows to {filename}\n")
        return True

    except requests.exceptions.Timeout:
        print("❌ Connection Timeout: The server took too long to reply. Try again later.")
    except requests.exceptions.ConnectionError:
        print("❌ Network Connection Error: Please verify your internet connection setup.")
    except Exception as e:
        print(f"❌ Unexpected Error occurred: {str(e)}")
    return False

# =====================================================================
# RUNNING THE FETCH PIPELINE
# =====================================================================

# Part A: Fetch the core HDFC Target Fund
fetch_and_save_nav(125497)

# Part B: Fetch the 5 Key Schemes loop
schemes = {
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_Large_Cap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for name, code in schemes.items():
    fetch_and_save_nav(code, name)

Fetching data for: 125497 (Code: 125497)...
✅ Success! Saved 3108 historical rows to data/raw/hdfc_125497_raw.csv

Fetching data for: SBI_Bluechip (Code: 119551)...
❌ Connection Timeout: The server took too long to reply. Try again later.
Fetching data for: ICICI_Bluechip (Code: 120503)...
✅ Success! Saved 3324 historical rows to data/raw/nav_ICICI_Bluechip_120503.csv

Fetching data for: Nippon_Large_Cap (Code: 118632)...
✅ Success! Saved 3315 historical rows to data/raw/nav_Nippon_Large_Cap_118632.csv

Fetching data for: Axis_Bluechip (Code: 119092)...
✅ Success! Saved 3582 historical rows to data/raw/nav_Axis_Bluechip_119092.csv

Fetching data for: Kotak_Bluechip (Code: 120841)...
✅ Success! Saved 3318 historical rows to data/raw/nav_Kotak_Bluechip_120841.csv



In [2]:
import os
import requests
import pandas as pd

# 1. Create the target directory if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

# 2. Use the optimized/latest endpoint for SBI Bluechip to bypass the timeout
url = "https://api.mfapi.in/mf/119551/latest"
print("Fetching SBI Bluechip data...")

try:
    response = requests.get(url, timeout=15)
    
    if response.status_code == 200:
        payload = response.json()
        
        # 3. Parse and convert to a DataFrame
        df = pd.DataFrame(payload['data'])
        df['amfi_code'] = 119551
        df['scheme_name'] = "SBI_Bluechip"
        
        # 4. Save to your raw data directory
        output_path = "data/raw/nav_SBI_Bluechip_119551.csv"
        df.to_csv(output_path, index=False)
        print(f"✅ Success! Saved SBI Bluechip data to {output_path}")
    else:
        print(f"❌ Failed to fetch. Server returned status code: {response.status_code}")

except requests.exceptions.Timeout:
    print("❌ Connection Timeout: The API server is heavily overloaded. Try running this code again in a few minutes.")
except Exception as e:
    print(f"❌ An error occurred: {e}")
    

Fetching SBI Bluechip data...
❌ Connection Timeout: The API server is heavily overloaded. Try running this code again in a few minutes.


In [4]:
import os
import requests
import pandas as pd

# 1. Create the target directory if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

# 2. Use the optimized/latest endpoint for SBI Bluechip to bypass the timeout
url = "https://api.mfapi.in/mf/119551/latest"
print("Fetching SBI Bluechip data...")

try:
    response = requests.get(url, timeout=15)
    
    if response.status_code == 200:
        payload = response.json()
        
        # 3. Parse and convert to a DataFrame
        df = pd.DataFrame(payload['data'])
        df['amfi_code'] = 119551
        df['scheme_name'] = "SBI_Bluechip"
        
        # 4. Save to your raw data directory
        output_path = "data/raw/nav_SBI_Bluechip_119551.csv"
        df.to_csv(output_path, index=False)
        print(f"✅ Success! Saved SBI Bluechip data to {output_path}")
    else:
        print(f"❌ Failed to fetch. Server returned status code: {response.status_code}")

except requests.exceptions.Timeout:
    print("❌ Connection Timeout: The API server is heavily overloaded. Try running this code again in a few minutes.")
except Exception as e:
    print(f"❌ An error occurred: {e}")
    

Fetching SBI Bluechip data...
✅ Success! Saved SBI Bluechip data to data/raw/nav_SBI_Bluechip_119551.csv


In [13]:
import os

raw_dir = "data/raw"

if not os.path.exists(raw_dir):
    print(f"❌ The directory '{raw_dir}' does not even exist! Please create it.")
else:
    print(f"📂 Current files inside '{raw_dir}':")
    files = os.listdir(raw_dir)
    if not files:
        print("   (The folder is completely empty!)")
    for file in files:
        print(f"   - {file}")
        

📂 Current files inside 'data/raw':
   - hdfc_125497_raw.csv
   - nav_Axis_Bluechip_119092.csv
   - nav_ICICI_Bluechip_120503.csv
   - nav_Kotak_Bluechip_120841.csv
   - nav_Nippon_Large_Cap_118632.csv
   - nav_SBI_Bluechip_119551.csv


In [14]:
import pandas as pd
import glob
import os

# Set path based on where you are running your file
raw_folder = "data/raw"  # Use "../data/raw" if you are inside the notebooks folder

# Find all files ending in .csv inside the folder
csv_files = glob.glob(os.path.join(raw_folder, "*.csv"))

# Create a dictionary to store our datasets
all_datasets = {}

print(f"--- Scanning and Loading all datasets in {raw_folder} ---")
for file_path in csv_files:
    # Extract just the name of the file without the '.csv' extension to use as a key
    file_name = os.path.basename(file_path).replace(".csv", "")
    
    # Load into pandas and save in our dictionary
    all_datasets[file_name] = pd.read_csv(file_path)
    print(f"Loaded dataset key: '{file_name}' | Shape: {all_datasets[file_name].shape}")

print("\n🎉 Done! You can now access any file like this: all_datasets['fund_master']")

--- Scanning and Loading all datasets in data/raw ---
Loaded dataset key: 'hdfc_125497_raw' | Shape: (3108, 3)
Loaded dataset key: 'nav_Axis_Bluechip_119092' | Shape: (3582, 4)
Loaded dataset key: 'nav_ICICI_Bluechip_120503' | Shape: (3324, 4)
Loaded dataset key: 'nav_Kotak_Bluechip_120841' | Shape: (3318, 4)
Loaded dataset key: 'nav_Nippon_Large_Cap_118632' | Shape: (3315, 4)
Loaded dataset key: 'nav_SBI_Bluechip_119551' | Shape: (1, 4)

🎉 Done! You can now access any file like this: all_datasets['fund_master']


In [15]:
import os

raw_dir = "data/raw"

if not os.path.exists(raw_dir):
    print(f"❌ The directory '{raw_dir}' does not even exist! Please create it.")
else:
    print(f"📂 Current files inside '{raw_dir}':")
    files = os.listdir(raw_dir)
    if not files:
        print("   (The folder is completely empty!)")
    for file in files:
        print(f"   - {file}")

📂 Current files inside 'data/raw':
   - hdfc_125497_raw.csv
   - nav_Axis_Bluechip_119092.csv
   - nav_ICICI_Bluechip_120503.csv
   - nav_Kotak_Bluechip_120841.csv
   - nav_Nippon_Large_Cap_118632.csv
   - nav_SBI_Bluechip_119551.csv


In [4]:
import os
import glob
import pandas as pd

raw_dir = "C:/Users/ASUS/Documents/College folders/Bluestock internship/Mutual_Fund_Analytics/data/raw"
print("--- Day 1: Step 4 Validation Pipeline ---")

# 1. Look for fund_master.csv
fund_master_path = os.path.join(raw_dir, "fund_master.csv")
if os.path.exists(fund_master_path):
    df_master = pd.read_csv(fund_master_path)
    print("✅ Successfully loaded fund_master.csv")
else:
    print(f"❌ Error: 'fund_master.csv' is missing from {raw_dir}.")
    print("👉 Action: Find your provided master dataset, rename it to 'fund_master.csv', and drop it in that folder.")
    df_master = None

# 2. Look for nav_history.csv OR combine your API downloads
nav_history_path = os.path.join(raw_dir, "nav_history.csv")
df_nav_history = None

if os.path.exists(nav_history_path):
    df_nav_history = pd.read_csv(nav_history_path)
    print("✅ Successfully loaded nav_history.csv")
else:
    print(f"ℹ️ 'nav_history.csv' not found. Checking for individual API files downloaded in Step 3...")
    # Find all nav files you grabbed from the API
    api_files = glob.glob(os.path.join(raw_dir, "nav_*.csv"))
    
    if api_files:
        print(f"Found {len(api_files)} API data files. Merging them automatically into one dataset...")
        list_of_dfs = []
        for file in api_files:
            temp_df = pd.read_csv(file)
            list_of_dfs.append(temp_df)
        
        df_nav_history = pd.concat(list_of_dfs, ignore_index=True)
        # Ensure the combined file has an amfi_code column for verification
        if 'amfi_code' not in df_nav_history.columns:
            df_nav_history['amfi_code'] = df_nav_history['scheme_code'] if 'scheme_code' in df_nav_history.columns else None
            
        print("✅ Successfully created combined historical dataset from API files.")
    else:
        print("❌ Error: No historical NAV data files found at all.")

# 3. Cross-Validate AMFI codes if both datasets are ready
if df_master is not None and df_nav_history is not None:
    # Standardize column names to lowercase to prevent case-sensitive crashes
    df_master.columns = df_master.columns.str.lower()
    df_nav_history.columns = df_nav_history.columns.str.lower()
    
    if 'amfi_code' in df_master.columns and 'amfi_code' in df_nav_history.columns:
        master_codes = set(df_master['amfi_code'].dropna().unique())
        nav_codes = set(df_nav_history['amfi_code'].dropna().unique())
        
        missing_codes = master_codes - nav_codes
        
        print("\n=== DATA QUALITY CONFORMANCE REPORT ===")
        print(f"Total Unique Funds in Master: {len(master_codes)}")
        print(f"Total Unique Funds in History: {len(nav_codes)}")
        
        if not missing_codes:
            print("🎉 Validation Passed: Every AMFI code match matches perfectly!")
        else:
            print(f"⚠️ Validation Warning: {len(missing_codes)} codes in your master list don't have matching historical files.")
            print("Missing codes list:", missing_codes)
            
        # Optional: Save this combined history file so Day 2 can use it easily
        if not os.path.exists(nav_history_path):
            df_nav_history.to_csv(nav_history_path, index=False)
            print(f"💾 Saved combined dataset as '{nav_history_path}' for Day 2 usage.")
    else:
        print("❌ Mapping Failed: Make sure both CSV files contain an 'amfi_code' column header.")

--- Day 1: Step 4 Validation Pipeline ---
✅ Successfully loaded fund_master.csv
✅ Successfully loaded nav_history.csv

=== DATA QUALITY CONFORMANCE REPORT ===
Total Unique Funds in Master: 40
Total Unique Funds in History: 40
🎉 Validation Passed: Every AMFI code match matches perfectly!
